In [2]:
import pandas as pd
from io import StringIO
import json
import dash
from dash import html, dcc, dash_table
import plotly.express as px
from dash.dependencies import Input, Output

In [3]:
price_cols = ["bid_price_1", "bid_price_2", "bid_price_3", "ask_price_1", "ask_price_2", "ask_price_3"]
volume_cols = ["bid_volume_1", "bid_volume_2", "bid_volume_3", "ask_volume_1", "ask_volume_2", "ask_volume_3"]


#If using log data use this, make sure to change filename here and add file to current folder

with open("5831.log") as file:
    json_data = json.load(file)
#prices = pd.read_csv("prices_round_0_day_-1.csv", sep=';')
#trades = pd.read_csv("trades_round_0_day_-1.csv", sep=';')

#If using data from datapacket off website use this
prices = pd.read_csv(StringIO(json_data["activitiesLog"]), sep=';')
trades = pd.DataFrame(json_data["tradeHistory"])

trades["our_direction"]=trades.apply(lambda row: "buy" if row["buyer"]=="SUBMISSION" else "sell" if row["seller"]=="SUBMISSION" else "bot_trade", axis=1) 
trades["buyer"] = trades["buyer"].fillna("")
trades["seller"] = trades["seller"].fillna("")

eme_prices = prices[prices["product"]=="EMERALDS"]
tom_prices = prices[prices["product"]=="TOMATOES"]

eme_trades = trades[trades["symbol"]=="EMERALDS"]
tom_trades = trades[trades["symbol"]=="TOMATOES"]




#changes prices data into long format
def longDF(df):
    quotes=[]
    for p_col, v_col in zip(price_cols, volume_cols):
        data = df[["timestamp"]].copy()
        data["price"] = df[p_col].values
        data["volume"] = df[v_col].values
        if "bid" in p_col:
            data["side"] = "bid"
        if "ask" in p_col:
            data["side"] = "ask"
        quotes.append(data)
    return pd.concat(quotes, ignore_index=True)

tom_long=longDF(tom_prices)
tom_long.dropna(inplace=True)
eme_long=longDF(eme_prices)
eme_long.dropna(inplace=True)

symbol_map= {"buy": "triangle-up", "sell": "triangle-down", "bot_trade": "star"}
datasets = {"EMERALDS": (eme_long, eme_trades), "TOMATOES": (tom_long, tom_trades)}

#generates scatter plot graphs
def make_figure(ticker):
    prices, trades = datasets[ticker]
    figure=px.scatter(prices, x="timestamp", y="price", size="volume", color="side")
    figure.add_scatter(x=trades["timestamp"], y=trades["price"], mode="markers", name="trades",
                       customdata=trades[["timestamp","quantity", "buyer", "seller"]].values,
                       hovertemplate="timestamp=%{x}<br>price=%{y}<br>volume=%{customdata[1]}<br>buyer=%{customdata[2]}<br>seller=%{customdata[3]}<extra></extra>",
                       marker_symbol=trades["our_direction"].map(symbol_map), marker_size=20, marker_color="black")
    return figure




def grapher():
    app = dash.Dash(__name__)
    
    app.layout = html.Div([
        dcc.Dropdown(id="ticker_dropdown", options=[{"label": x, "value": x} for x in datasets.keys()],
                     value="EMERALDS"),
        dcc.Graph(id="main_graph"),
    ])
    @app.callback(
        Output("main_graph", "figure"),
        Input("ticker_dropdown", "value"))
    def update_graph(selected_ticker):
        return make_figure(selected_ticker)
    
    app.run(debug=True)
grapher()

FileNotFoundError: [Errno 2] No such file or directory: '5831.log'